# Personalized Real Estate Agent

## Step1: Setting Up the Python Application

In [1]:
# import os

# os.environ["OPENAI_API_KEY"] = "YOUR API KEY"
# os.environ["OPENAI_API_BASE"] = "https://openai.vocareum.com/v1"

%load_ext dotenv
%dotenv

from uuid import uuid4
from typing import List
import json

from langchain_core.documents import Document
from langchain.vectorstores import Chroma
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field, NonNegativeInt

## Step 2: Generating Real Estate Listings

In [2]:
model_name = "gpt-3.5-turbo"
temperature = 0.0
llm = OpenAI(model_name=model_name, temperature=temperature, max_tokens = 4000)

/home/eseidinger/dev/genai-nano/venv-rea/lib/python3.12/site-packages/langchain/llms/openai.py:202: UserWarning: You are trying to use a chat model. This way of initializing it is no longer supported. Instead, please use: `from langchain.chat_models import ChatOpenAI`
  warnings.warn(
/home/eseidinger/dev/genai-nano/venv-rea/lib/python3.12/site-packages/langchain/llms/openai.py:790: UserWarning: You are trying to use a chat model. This way of initializing it is no longer supported. Instead, please use: `from langchain.chat_models import ChatOpenAI`
  warnings.warn(


In [3]:
class RealEstateListing(BaseModel):
    neighborhood: str = Field(description="neighborhood of the listing")
    price: str = Field(description="price of the listing")
    bedrooms: NonNegativeInt = Field(description="number of bedrooms in the listing")
    bathrooms: NonNegativeInt = Field(description="number of bathrooms in the listing")
    house_size: str = Field(description="size of the house in sqft")
    description: str = Field(description="description of the listing")
    neighborhood_description: str = Field(description="description of the neighborhood")

class RealEstateListings(BaseModel):
    listings: List[RealEstateListing] = Field(description="list of real estate listings")
        
parser = PydanticOutputParser(pydantic_object=RealEstateListings)
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"listings": {"title": "Listings", "description": "list of real estate listings", "type": "array", "items": {"$ref": "#/definitions/RealEstateListing"}}}, "required": ["listings"], "definitions": {"RealEstateListing": {"title": "RealEstateListing", "type": "object", "properties": {"neighborhood": {"title": "Neighborhood", "description": "neighborhood of the listing", "type": "string"}, "price": {"title": "Price", "description": "price of the listing", "type": "string"}, "bedrooms": {"title": "Bedrooms", "description": "number of

In [4]:
def generate_real_estate_listings(num_listings: int) -> RealEstateListings:
    prompt = PromptTemplate(
        template="{question}\n{format_instructions}\nGenerate {num_listings} listings.",
        input_variables=["question", "num_listings"],
        partial_variables={"format_instructions": parser.get_format_instructions},
    )

    question = """
Generate real estate listings. An example listing can be:

Neighborhood: Green Oaks
Price: $800,000
Bedrooms: 3
Bathrooms: 2
House Size: 2,000 sqft

Description: Welcome to this eco-friendly oasis nestled in the heart of Green Oaks. This charming 3-bedroom, 2-bathroom home boasts energy-efficient features such as solar panels and a well-insulated structure. Natural light floods the living spaces, highlighting the beautiful hardwood floors and eco-conscious finishes. The open-concept kitchen and dining area lead to a spacious backyard with a vegetable garden, perfect for the eco-conscious family. Embrace sustainable living without compromising on style in this Green Oaks gem.

Neighborhood Description: Green Oaks is a close-knit, environmentally-conscious community with access to organic grocery stores, community gardens, and bike paths. Take a stroll through the nearby Green Oaks Park or grab a cup of coffee at the cozy Green Bean Cafe. With easy access to public transportation and bike lanes, commuting is a breeze.
    """

    query = prompt.format(question = question, num_listings = num_listings)
    output = llm.predict(query)
    result = parser.parse(output)
    return result

In [5]:
listings = generate_real_estate_listings(10)
print(listings)

listings=[RealEstateListing(neighborhood='Green Oaks', price='$800,000', bedrooms=3, bathrooms=2, house_size='2,000 sqft', description='Welcome to this eco-friendly oasis nestled in the heart of Green Oaks. This charming 3-bedroom, 2-bathroom home boasts energy-efficient features such as solar panels and a well-insulated structure. Natural light floods the living spaces, highlighting the beautiful hardwood floors and eco-conscious finishes. The open-concept kitchen and dining area lead to a spacious backyard with a vegetable garden, perfect for the eco-conscious family. Embrace sustainable living without compromising on style in this Green Oaks gem.', neighborhood_description='Green Oaks is a close-knit, environmentally-conscious community with access to organic grocery stores, community gardens, and bike paths. Take a stroll through the nearby Green Oaks Park or grab a cup of coffee at the cozy Green Bean Cafe. With easy access to public transportation and bike lanes, commuting is a b

In [6]:
with open("listings.txt", "w", encoding="utf-8") as f:
    f.write(json.dumps(listings.dict(), indent=4))

## Step 3: Storing Listings in a Vector Database

In [7]:
listing2Document = lambda listing: Document(
    page_content = f"""
    Neighborhood: {listing.neighborhood}
    Price: {listing.price}
    Bedrooms: {listing.bedrooms}
    Bathrooms: {listing.bathrooms}
    House Size: {listing.house_size}

    Description: {listing.description}

    Neighborhood Description: {listing.neighborhood_description}
    """,
    id = str(uuid4())
)

documents = [listing2Document(listing) for listing in listings.listings]

db = Chroma.from_documents(documents=documents, embedding=OpenAIEmbeddings())

## Step 4: Building the User Preference Interface

In [8]:
questions = [   
    "How big do you want your house to be?",
    "What are 3 most important things for you in choosing this property?", 
    "Which amenities would you like?", 
    "Which transportation options are important to you?",
    "How urban do you want your neighborhood to be?",   
]
answers = [
    "A comfortable three-bedroom house with a spacious kitchen and a cozy living room.",
    "A quiet neighborhood, good local schools, and convenient shopping options.",
    "A backyard for gardening, a two-car garage, and a modern, energy-efficient heating system.",
    "Easy access to a reliable bus line, proximity to a major highway, and bike-friendly roads.",
    "A balance between suburban tranquility and access to urban amenities like restaurants and theaters."
]

## Step 5: Searching Based on Preferences

In [9]:
query = "\n".join(answers)

search_results = db.similarity_search(query, k=3)

for i, result in enumerate(search_results):
    print(f"\n\nResult {i+1}")
    print(result.page_content)



Result 1

    Neighborhood: Green Oaks
    Price: $800,000
    Bedrooms: 3
    Bathrooms: 2
    House Size: 2,000 sqft

    Description: Welcome to this eco-friendly oasis nestled in the heart of Green Oaks. This charming 3-bedroom, 2-bathroom home boasts energy-efficient features such as solar panels and a well-insulated structure. Natural light floods the living spaces, highlighting the beautiful hardwood floors and eco-conscious finishes. The open-concept kitchen and dining area lead to a spacious backyard with a vegetable garden, perfect for the eco-conscious family. Embrace sustainable living without compromising on style in this Green Oaks gem.

    Neighborhood Description: Green Oaks is a close-knit, environmentally-conscious community with access to organic grocery stores, community gardens, and bike paths. Take a stroll through the nearby Green Oaks Park or grab a cup of coffee at the cozy Green Bean Cafe. With easy access to public transportation and bike lanes, commuting 

## Step 6: Personalizing Listing Descriptions

In [10]:
questionsAndAnswers2Preferences = lambda questions, answers: "\n".join([f"{question}\n{answer}" for question, answer in zip(questions, answers)])

preferences = questionsAndAnswers2Preferences(questions, answers)

def create_personalized_description(preferences: str, listing: str) -> str:
    prompt = PromptTemplate(
        template="""
        Based on the following preferences:
        {preferences}
        
        Augment the following listing to resonate with the preferences, emphasizing the features that align with the preferences.
        {listing}

        Personalized Description:
        """,
        input_variables=["preferences", "listing"]
    )

    query = prompt.format(preferences = preferences, listing = listing)
    output = llm.predict(query)
    return output

for i, result in enumerate(search_results):
    output = create_personalized_description(answers[i], result.page_content)
    print(f"\n\nPersonalized Description {i+1}")
    print(output)



Personalized Description 1
Welcome to your dream home in the eco-friendly neighborhood of Green Oaks. This comfortable 3-bedroom, 2-bathroom house is the perfect blend of sustainability and style. With a spacious kitchen and cozy living room, this 2,000 sqft home is ideal for those who value both comfort and eco-conscious living.

Step inside to discover the natural light that fills the living spaces, showcasing the beautiful hardwood floors and eco-conscious finishes throughout. The open-concept kitchen and dining area lead to a spacious backyard with a vegetable garden, perfect for those who appreciate sustainable living.

Located in the close-knit community of Green Oaks, you'll have access to organic grocery stores, community gardens, and bike paths. Take a leisurely stroll through Green Oaks Park or enjoy a cup of coffee at the nearby Green Bean Cafe. With easy access to public transportation and bike lanes, commuting is a breeze in this environmentally-conscious neighborhood.

